# LLM From Scratch

Trains a small GPT-style language model on two ML textbooks extracted from PDF.

## 1. Load & Clean Text from PDFs

In [44]:
import pymupdf

pdf_configs = [
    {"path": "./resources/Hands-On Large Language Models.pdf", "start": 23, "end": 411},
    {"path": "./resources/Introducing-MLOps.pdf", "start": 15, "end": 171},
]

raw_text = ""
for cfg in pdf_configs:
    doc = pymupdf.open(cfg["path"])
    for page in doc:
        if page.number < cfg["start"] or page.number > cfg["end"]:
            continue
        for block in page.get_text("blocks"):
            y1 = block[3]
            if y1 < page.rect.height * 0.92:  # skip footer region
                raw_text += block[4]
    doc.close()

print(f"Total characters: {len(raw_text)}")
print(raw_text[:200])

Total characters: 877108
CHAPTER 1
An Introduction to Large Language Models
Humanity is at an inflection point. From 2012 onwards, developments in building AI
systems (using deep neural networks) accelerated so that by the en


## 2. Dataset & DataLoader

In [45]:
import tiktoken
import torch
from torch.utils.data import DataLoader, Dataset


class GPTDataset(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        self.input_ids, self.target_ids = [], []
        for i in range(0, len(token_ids) - max_length, stride):
            self.input_ids.append(torch.tensor(token_ids[i : i + max_length]))
            self.target_ids.append(torch.tensor(token_ids[i + 1 : i + max_length + 1]))
        print(f"Tokens: {len(token_ids):,} → {len(self.input_ids):,} samples")

    def __len__(self): return len(self.input_ids)
    def __getitem__(self, i): return self.input_ids[i], self.target_ids[i]


def create_dataloader(txt, batch_size=16, max_length=256, stride=128,
                      shuffle=True, drop_last=True):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDataset(txt, tokenizer, max_length, stride)
    return DataLoader(dataset, batch_size=batch_size,
                      shuffle=shuffle, drop_last=drop_last, num_workers=0)


# 90/10 train-val split
split = int(len(raw_text) * 0.9)
train_loader = create_dataloader(raw_text[:split], batch_size=16,
                                  max_length=256, stride=128, shuffle=True)
val_loader   = create_dataloader(raw_text[split:], batch_size=16,
                                  max_length=256, stride=128, shuffle=False)

Tokens: 193,564 → 1,511 samples
Tokens: 18,559 → 143 samples


## 3. GPT Model

A decoder-only transformer: token + positional embeddings → N transformer blocks → logits over vocab.

In [46]:
import torch.nn as nn


# ── Hyperparameters ────────────────────────────────────────────────────────
GPT_CONFIG = {
    "vocab_size":     50257,   # GPT-2 BPE vocabulary
    "context_length": 256,     # max tokens the model sees at once
    "emb_dim":        256,     # embedding / hidden dimension
    "n_heads":        4,       # number of attention heads
    "n_layers":       4,       # number of transformer blocks
    "drop_rate":      0.1,     # dropout probability
    "qkv_bias":       False,   # no bias in Q/K/V projections (GPT-2 style)
}


class MultiHeadAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_heads  = cfg["n_heads"]
        self.head_dim = cfg["emb_dim"] // cfg["n_heads"]

        self.W_q = nn.Linear(cfg["emb_dim"], cfg["emb_dim"], bias=cfg["qkv_bias"])
        self.W_k = nn.Linear(cfg["emb_dim"], cfg["emb_dim"], bias=cfg["qkv_bias"])
        self.W_v = nn.Linear(cfg["emb_dim"], cfg["emb_dim"], bias=cfg["qkv_bias"])
        self.out_proj = nn.Linear(cfg["emb_dim"], cfg["emb_dim"])
        self.dropout  = nn.Dropout(cfg["drop_rate"])

        # causal mask: upper triangle = -inf so future tokens are invisible
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(cfg["context_length"], cfg["context_length"]), diagonal=1)
        )

    def forward(self, x):
        B, T, C = x.shape

        def split_heads(t):
            return t.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        Q, K, V = split_heads(self.W_q(x)), split_heads(self.W_k(x)), split_heads(self.W_v(x))

        scores = Q @ K.transpose(-2, -1) / self.head_dim ** 0.5
        scores = scores.masked_fill(self.mask[:T, :T].bool(), float("-inf"))
        weights = self.dropout(torch.softmax(scores, dim=-1))

        out = (weights @ V).transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(out)


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            nn.GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
            nn.Dropout(cfg["drop_rate"]),
        )

    def forward(self, x): return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attn = MultiHeadAttention(cfg)
        self.ff   = FeedForward(cfg)
        self.ln1  = nn.LayerNorm(cfg["emb_dim"])
        self.ln2  = nn.LayerNorm(cfg["emb_dim"])

    def forward(self, x):
        x = x + self.attn(self.ln1(x))   # pre-norm + residual
        x = x + self.ff(self.ln2(x))
        return x


class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop    = nn.Dropout(cfg["drop_rate"])
        self.blocks  = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.ln_f    = nn.LayerNorm(cfg["emb_dim"])
        self.head    = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, idx):
        B, T = idx.shape
        positions = torch.arange(T, device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(positions))
        x = self.blocks(x)
        return self.head(self.ln_f(x))


device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
model = GPT(GPT_CONFIG).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Device: {device}")
print(f"Parameters: {total_params:,}")

Device: mps
Parameters: 28,953,600


## 4. Training Loop

In [47]:
def calc_loss(model, loader, device, max_batches=10):
    model.eval()
    total = 0.0
    with torch.no_grad():
        for i, (x, y) in enumerate(loader):
            if i >= max_batches:
                break
            logits = model(x.to(device))
            # logits: [B, T, vocab] → flatten for cross_entropy
            loss = nn.functional.cross_entropy(
                logits.view(-1, logits.size(-1)), y.to(device).view(-1)
            )
            total += loss.item()
    model.train()
    return total / min(max_batches, i + 1)


def train(model, train_loader, val_loader, epochs=10, lr=3e-4):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.1)
    model.train()

    for epoch in range(1, epochs + 1):
        for step, (x, y) in enumerate(train_loader):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = nn.functional.cross_entropy(
                logits.view(-1, logits.size(-1)), y.view(-1)
            )
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        train_loss = calc_loss(model, train_loader, device)
        val_loss   = calc_loss(model, val_loader, device)
        print(f"Epoch {epoch:02d} | train loss {train_loss:.4f} | val loss {val_loss:.4f}")

    return model


model = train(model, train_loader, val_loader, epochs=10, lr=3e-4)

Epoch 01 | train loss 6.5191 | val loss 7.0867
Epoch 02 | train loss 5.8951 | val loss 6.6820
Epoch 03 | train loss 5.3566 | val loss 6.4834
Epoch 04 | train loss 5.0572 | val loss 6.3221
Epoch 05 | train loss 4.7679 | val loss 6.2539
Epoch 06 | train loss 4.4661 | val loss 6.2229
Epoch 07 | train loss 4.3169 | val loss 6.2004
Epoch 08 | train loss 4.1145 | val loss 6.1426
Epoch 09 | train loss 3.9140 | val loss 6.1220
Epoch 10 | train loss 3.7954 | val loss 6.1202


## 5. Text Generation

In [48]:
def generate(model, prompt, max_new_tokens=100, temperature=0.8, top_k=40):
    tokenizer = tiktoken.get_encoding("gpt2")
    model.eval()

    ids = tokenizer.encode(prompt)
    ids = torch.tensor(ids).unsqueeze(0).to(device)  # [1, T]

    with torch.no_grad():
        for _ in range(max_new_tokens):
            # crop to context window
            ids_crop = ids[:, -GPT_CONFIG["context_length"]:]
            logits = model(ids_crop)
            logits = logits[:, -1, :] / temperature  # last token logits

            # top-k filtering
            if top_k:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = float("-inf")

            probs = torch.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            ids = torch.cat([ids, next_id], dim=1)

    return tokenizer.decode(ids[0].tolist())


output = generate(model, prompt="Large language models are", max_new_tokens=150)
print(output)

Large language models are
of the model is
the first step is deployed.
the level of training examples is a
the performance is more than its
time is often referred to
the current model.
1, the organization, the business.
can be used to generate the best solution to be used to run. For example, if
models. This
Figure 10-tokens of language model is that the model’t have the model, this chapter, the model with the model.
Figure 1-based the model to the
models, the training examples of information.
and the next chapter is
Figure 3-9.
a-2. A model can be shown in the model for the model is more detail as
